In [1]:
import os

# Caminho absoluto da pasta onde o script está sendo executado
diretorio_atual = os.getcwd()

print("Diretório atual:", diretorio_atual)

Diretório atual: /workspaces/IC-RNA-2025/fuction_ensemble_3/media_ponderada


In [1]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics import mean_squared_error

# ================================
# 1) CONFIGURAÇÕES
# ================================
arquivo_pesos = "/workspaces/IC-RNA-2025/fuction_ensemble_3/media_ponderada/todos_resultados_50_lm1.xlsx"
diretorio_redes = "/workspaces/IC-RNA-2025/fuction_ensemble_3/media_ponderada/1000_model"
limiar_peso = 0.1

df_pesos = pd.read_excel(arquivo_pesos).head(50)

resultados = []

for idx, row in df_pesos.iterrows():
    # Coluna A = nomes das redes, Coluna E = pesos
    nomes_str = row.iloc[0]
    pesos_str = row.iloc[4]

    # 🔧 Converte string "[a b c]" → lista de strings → lista de floats
    nomes_redes = nomes_str.strip("[]").replace("'", "").split(",")
    nomes_redes = [nome.strip() for nome in nomes_redes]

    pesos = np.array(pesos_str.strip("[]").split(), dtype=float)


    # ===============================
    # 2) CARREGA Z E Z_pred
    # ===============================
    dfs = [pd.read_excel(f"{diretorio_redes}/{nome}") for nome in nomes_redes]
    Z = dfs[0]["Z"].values.reshape(-1, 1)
    Z_preds = np.hstack([df["Z_pred"].values.reshape(-1, 1) for df in dfs])

    # ===============================
    # 3) MSE COM TODAS AS REDES
    # ===============================
    Z_pred_ponderada = Z_preds @ pesos
    mse_sup = mean_squared_error(Z, Z_pred_ponderada)

    # ===============================
    # 4) REMOVE REDES COM PESO < 0.1
    # ===============================
    mask = pesos >= 0.1
    nomes_filtrados = [nome for nome, keep in zip(nomes_redes, mask) if keep]
    pesos_filtrados = pesos[mask]
    pesos_filtrados = pesos_filtrados / np.sum(pesos_filtrados)

    Z_preds_filtrados = Z_preds[:, mask]
    yhat_filtrado = Z_preds_filtrados @ pesos_filtrados
    mse_filtrado = mean_squared_error(Z, yhat_filtrado)

    resultados.append({
        "linha": idx + 1,
        "MSE_sup": mse_sup,
        "MSE_sup_novo": mse_filtrado,
        "num_redes_total": len(nomes_redes),
        "num_redes_filtradas": len(nomes_filtrados),
        "nome_redes_filtradas": nomes_filtrados
    })


# ===============================
# 5) RESULTADOS EM DATAFRAME
# ===============================
df_resultados = pd.DataFrame(resultados)
print(df_resultados)

# Salva se quiser
df_resultados.to_excel("comparacao_mse_50_ensembles.xlsx", index=False)

    linha      MSE_sup  MSE_sup_novo  num_redes_total  num_redes_filtradas  \
0       1  1509.210053   1512.183044               10                    2   
1       2  1509.915406   1509.931026               10                    2   
2       3  1510.220881   1510.330190               10                    2   
3       4  1510.499939   1510.588767               10                    2   
4       5  1510.794914   1512.149325               10                    2   
5       6  1513.723823   1513.911578               10                    2   
6       7  1525.521500   1530.179517               10                    2   
7       8  1531.893038   1533.865926               10                    2   
8       9  1532.789940   1536.723219               10                    2   
9      10  1533.236368   1545.031391               10                    2   
10     11  1537.067012   1548.075057               10                    2   
11     12  1542.051517   1553.223610               10           

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import os 

# ================================
# 1) CONFIGURAÇÕES
# ================================
diretorio_redes_1000 = "/workspaces/IC-RNA-2025/fuction_ensemble_3/media_ponderada/1000_model"
diretorio_redes_25 = "/workspaces/IC-RNA-2025/fuction_ensemble_3/media_ponderada/25_model"
arquivo_pesos = "/workspaces/IC-RNA-2025/fuction_ensemble_3/media_ponderada/todos_resultados_50_lm1_filtrada_covar.xlsx"
limiar_peso = 0.1

df_pesos = pd.read_excel(arquivo_pesos)

resultados = []

# ===============================
# FUNÇÃO AUXILIAR
# ===============================
def extrai_id(nome):
   
    return nome.split("model_")[1]


# ===============================
# FUNÇÃO DE AVALIAÇÃO COMPLETA
# ===============================
def avalia_filtrado(nomes_filtrados, pesos_filtrados):

    # ===============================
    # Reconstruindo nomes corretos
    # ===============================
    ids = [extrai_id(nome) for nome in nomes_filtrados]

    nomes_1000 = [f"1000_model_{id}" for id in ids]
    nomes_25 = [f"25_model_{id}" for id in ids]

    # ===============================
    # LEITURA 1000_model
    # ===============================
    df_list1 = []
    for nome in nomes_1000:
        caminho = os.path.join(diretorio_redes_1000, nome)
        if not os.path.exists(caminho):
            print("Arquivo não encontrado:", caminho)
            return None
        df_list1.append(pd.read_excel(caminho))

    z_preds_1000 = np.hstack([
        df['Z_pred'].values.reshape(-1, 1)
        for df in df_list1
    ])
    Z_1000 = df_list1[0]['Z'].values.reshape(-1, 1)

    # ===============================
    # LEITURA 25_model
    # ===============================
    df_list2 = []
    for nome in nomes_25:
        caminho = os.path.join(diretorio_redes_25, nome)
        if not os.path.exists(caminho):
            print("Arquivo não encontrado:", caminho)
            return None
        df_list2.append(pd.read_excel(caminho))

    z_preds_25 = np.hstack([
        df['Z_pred'].values.reshape(-1, 1)
        for df in df_list2
    ])
    Z_25 = df_list2[0]['Z'].values.reshape(-1, 1)

    # ===============================
    # ENSEMBLE
    # ===============================
    yhat_1000 = z_preds_1000 @ pesos_filtrados
    yhat_25 = z_preds_25 @ pesos_filtrados

    # ===============================
    # MÉTRICAS 1000 (superfície)
    # ===============================
    mse_1000 = np.mean((Z_1000.flatten() - yhat_1000.flatten()) ** 2)
    r2_1000 = r2_score(Z_1000, yhat_1000)

    # ===============================
    # MÉTRICAS 25 (bias-var-covar)
    # ===============================
    mse_25 = np.mean((Z_25.flatten() - yhat_25.flatten()) ** 2)
    r2_25 = r2_score(Z_25, yhat_25)

    M = z_preds_25.shape[1]

    var = np.mean([
        (z_preds_25[:, i] - yhat_25.flatten()) ** 2
        for i in range(M)
    ])

    bias = np.mean(yhat_25.flatten() - Z_25.flatten())

    cov_sum = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                cov_sum += np.mean(
                    (z_preds_25[:, i] - z_preds_25[:, i].mean()) *
                    (z_preds_25[:, j] - z_preds_25[:, j].mean())
                )

    covar = cov_sum / (M * (M - 1)) if M > 1 else 0

    return mse_1000, r2_1000, r2_25, mse_25, var, bias, covar


# ===============================
# LOOP PRINCIPAL
# ===============================
for idx, row in df_pesos.iterrows():

    nomes_str = row.iloc[0]
    pesos_str = row.iloc[4]

    nomes_redes = eval(nomes_str)  # aqui pode continuar usando

    pesos = np.fromstring(
        pesos_str.strip("[]"),
        sep=" "
    )

    # ===============================
    # FILTRAGEM
    # ===============================
    mask = pesos >= limiar_peso
    nomes_filtrados = [nome for nome, keep in zip(nomes_redes, mask) if keep]
    pesos_filtrados = pesos[mask]

    if len(pesos_filtrados) == 0:
        continue

    # Renormaliza
    pesos_filtrados = pesos_filtrados / np.sum(pesos_filtrados)

    # ===============================
    # AVALIAÇÃO
    # ===============================
    metricas = avalia_filtrado(nomes_filtrados, pesos_filtrados)

    if metricas is None:
        continue

    mse_1000_f, r2_1000_f, r2_25_f, mse_25_f, var, bias, covar = metricas

    resultados.append({
        "linha": idx + 1,
        "mse_1000_f": mse_1000_f,
        "r2_1000_f": r2_1000_f,
        "mse_25_f": mse_25_f,
        "r2_25_f": r2_25_f,
        "var_f": var,
        "bias_f": bias,
        "covar_f": covar,
        "num_redes_total": len(nomes_redes),
        "num_redes_filtradas": len(nomes_filtrados),
        "nomes_redes_filtradas": nomes_filtrados,
        "pesos_filtrados": pesos_filtrados
    })

# ===============================
# SALVA RESULTADOS
# ===============================
df_resultados = pd.DataFrame(resultados)
print(df_resultados)

df_resultados.to_excel("metricas_ensemble_filtrado.xlsx", index=False)

    linha   mse_1000_f  r2_1000_f    mse_25_f   r2_25_f       var_f    bias_f  \
0       1  1676.478438   0.995559  989.576571  0.997379    0.000000 -6.491815   
1       2  1676.478438   0.995559  989.576571  0.997379    0.000000 -6.491815   
2       3  1676.478438   0.995559  989.576571  0.997379    0.000000 -6.491815   
3       4  1676.478438   0.995559  989.576571  0.997379    0.000000 -6.491815   
4       5  1676.478438   0.995559  989.576571  0.997379    0.000000 -6.491815   
5       6  1676.478438   0.995559  989.576571  0.997379    0.000000 -6.491815   
6       7  1676.478438   0.995559  989.576571  0.997379    0.000000 -6.491815   
7       8  1676.478438   0.995559  989.576571  0.997379    0.000000 -6.491815   
8       9  1676.478438   0.995559  989.576571  0.997379    0.000000 -6.491815   
9      10  1676.478438   0.995559  989.576571  0.997379    0.000000 -6.491815   
10     11  1676.478438   0.995559  989.576571  0.997379    0.000000 -6.491815   
11     12  1676.478438   0.9

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import os

diretorio_redes_1000 = "/workspaces/IC-RNA-2025/fuction_ensemble_3/media_ponderada/1000_model"
diretorio_redes_25 = "/workspaces/IC-RNA-2025/fuction_ensemble_3/media_ponderada/25_model"
arquivo_pesos = "/workspaces/IC-RNA-2025/fuction_ensemble_3/media_ponderada/todos_resultados_50_lm1_filtrada_covar.xlsx"
limiar_peso = 0.1

df_pesos = pd.read_excel(arquivo_pesos).head(50)

resultados = []

# ===============================
# FUNÇÃO AUXILIAR
# ===============================
def extrai_id(nome):
    return nome.split("model_")[1]


# ===============================
# FUNÇÃO DE AVALIAÇÃO COMPLETA (FILTRADO)
# ===============================
def avalia_filtrado(nomes_filtrados, pesos_filtrados):

    ids = [extrai_id(nome) for nome in nomes_filtrados]

    nomes_1000 = [f"1000_model_{id}" for id in ids]
    nomes_25 = [f"25_model_{id}" for id in ids]

    # ---------- 1000 ----------
    df_list1 = [pd.read_excel(os.path.join(diretorio_redes_1000, nome)) for nome in nomes_1000]
    z_preds_1000 = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list1])
    Z_1000 = df_list1[0]['Z'].values.reshape(-1, 1)

    # ---------- 25 ----------
    df_list2 = [pd.read_excel(os.path.join(diretorio_redes_25, nome)) for nome in nomes_25]
    z_preds_25 = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list2])
    Z_25 = df_list2[0]['Z'].values.reshape(-1, 1)

    yhat_1000 = z_preds_1000 @ pesos_filtrados
    yhat_25 = z_preds_25 @ pesos_filtrados

    mse_1000 = np.mean((Z_1000.flatten() - yhat_1000.flatten()) ** 2)
    r2_1000 = r2_score(Z_1000, yhat_1000)

    mse_25 = np.mean((Z_25.flatten() - yhat_25.flatten()) ** 2)
    r2_25 = r2_score(Z_25, yhat_25)

    M = z_preds_25.shape[1]

    var = np.mean([(z_preds_25[:, i] - yhat_25.flatten()) ** 2 for i in range(M)])
    bias = np.mean(yhat_25.flatten() - Z_25.flatten())

    cov_sum = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                cov_sum += np.mean(
                    (z_preds_25[:, i] - z_preds_25[:, i].mean()) *
                    (z_preds_25[:, j] - z_preds_25[:, j].mean())
                )

    covar = cov_sum / (M * (M - 1)) if M > 1 else 0

    return mse_1000, r2_1000, r2_25, mse_25, var, bias, covar


# ===============================
# LOOP PRINCIPAL
# ===============================
for idx, row in df_pesos.iterrows():

    nomes_str = row.iloc[0]
    pesos_str = row.iloc[4]

    nomes_redes = eval(nomes_str)

    pesos = np.fromstring(
        pesos_str.strip("[]"),
        sep=" "
    )

    # ===============================
    # 🔵 MSE 1000 ORIGINAL (SEM FILTRO)
    # ===============================
    ids_orig = [extrai_id(nome) for nome in nomes_redes]
    nomes_1000_orig = [f"1000_model_{id}" for id in ids_orig]

    df_list_orig = [
        pd.read_excel(os.path.join(diretorio_redes_1000, nome))
        for nome in nomes_1000_orig
    ]

    z_preds_1000_orig = np.hstack([
        df['Z_pred'].values.reshape(-1, 1)
        for df in df_list_orig
    ])

    Z_1000_orig = df_list_orig[0]['Z'].values.reshape(-1, 1)

    yhat_1000_orig = z_preds_1000_orig @ pesos
    mse_1000_original = np.mean(
        (Z_1000_orig.flatten() - yhat_1000_orig.flatten()) ** 2
    )

    # ===============================
    # FILTRAGEM
    # ===============================
    mask = pesos >= limiar_peso
    nomes_filtrados = [nome for nome, keep in zip(nomes_redes, mask) if keep]
    pesos_filtrados = pesos[mask]

    if len(pesos_filtrados) == 0:
        continue

    pesos_filtrados = pesos_filtrados / np.sum(pesos_filtrados)

    # ===============================
    # AVALIAÇÃO FILTRADA
    # ===============================
    mse_1000_f, r2_1000_f, r2_25_f, mse_25_f, var, bias, covar = \
        avalia_filtrado(nomes_filtrados, pesos_filtrados)

    resultados.append({
        "linha": idx + 1,
        "mse_1000_original": mse_1000_original,   
        "mse_1000_f": mse_1000_f,
        "r2_1000_f": r2_1000_f,
        "mse_25_f": mse_25_f,
        "r2_25_f": r2_25_f,
        "var_f": var,
        "bias_f": bias,
        "covar_f": covar,
        "num_redes_total": len(nomes_redes),
        "num_redes_filtradas": len(nomes_filtrados),
        "nomes_redes_filtradas": nomes_filtrados,
        "pesos_filtrados": pesos_filtrados
    })

# ===============================
# SALVA RESULTADOS
# ===============================
df_resultados = pd.DataFrame(resultados)
print(df_resultados)

df_resultados.to_excel("metricas_ensemble_filtrado.xlsx", index=False)

    linha  mse_1000_original   mse_1000_f  r2_1000_f    mse_25_f   r2_25_f  \
0       1        1668.636864  1676.478438   0.995559  989.576571  0.997379   
1       2        1669.398721  1676.478438   0.995559  989.576571  0.997379   
2       3        1669.404393  1676.478438   0.995559  989.576571  0.997379   
3       4        1675.555052  1676.478438   0.995559  989.576571  0.997379   
4       5        1676.359213  1676.478438   0.995559  989.576571  0.997379   
5       6        1676.359213  1676.478438   0.995559  989.576571  0.997379   
6       7        1676.423634  1676.478438   0.995559  989.576571  0.997379   
7       8        1676.423634  1676.478438   0.995559  989.576571  0.997379   
8       9        1676.423634  1676.478438   0.995559  989.576571  0.997379   
9      10        1676.431804  1676.478438   0.995559  989.576571  0.997379   
10     11        1676.431804  1676.478438   0.995559  989.576571  0.997379   
11     12        1676.431804  1676.478438   0.995559  989.576571